## 模型稳定性测试：Lasso vs Ridge vs Sigmoid

测试目标：通过随机删除 3~5 口井，重复多次训练，比较三种模型的参数稳定性。


In [ ]:
# 导入依赖
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath("../"))

from src.data_utils import (
    extract_seismic_attributes_for_wells,
    filter_anomalous_attributes,
    filter_outlier_wells,
    filter_seismic_by_wells,
    identify_attributes,
    parse_petrel_file,
    preprocess_features,
)
from src.pca_analysis import perform_pca_analysis
from src.sigmoid import SigmoidModel

# 设置中文字体
plt.rcParams["font.family"] = "SimHei"
plt.rcParams["axes.unicode_minus"] = False

# 固定随机种子，保证可复现
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

### 实验配置


In [ ]:
# ==================== 实验配置 ====================
SURFACE_NAME = "H6-2"
WELL_FILE = "well_horizon_processed.xlsx"

# 稳定性测试参数
N_ITERATIONS = 50  # 总重复次数
K_RANGE = [3, 4, 5]  # 每次删除的井数（均匀随机选择）

# Ridge/Lasso 固定超参数（先用全量数据确定，后续固定）
RIDGE_ALPHA = 1.0
LASSO_ALPHA = 0.1

# Sigmoid 虚拟点配置（固定，不使用随机噪声）
VIRTUAL_CONFIG = {
    "placement_strategy": "conservative",
    "n_points": 10,
    "noise_factor": 0.0,  # 关闭噪声，保证可复现
    "auto_detect": True,
}

# 输出目录
output_dir = f"{SURFACE_NAME.replace('-', '_')}_stability_test"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"实验配置:")
print(f"  层位: {SURFACE_NAME}")
print(f"  重复次数: {N_ITERATIONS}")
print(f"  删井数量: {K_RANGE}")
print(f"  Ridge alpha: {RIDGE_ALPHA}")
print(f"  Lasso alpha: {LASSO_ALPHA}")
print(f"  输出目录: {output_dir}")

### 数据准备（与原流程一致，只执行一次）


In [ ]:
# 加载数据
data_dir = "../data"
data_seismic_url = os.path.join(data_dir, SURFACE_NAME)
data_seismic_attr = parse_petrel_file(data_seismic_url)
data_well_position = pd.read_excel(os.path.join(data_dir, WELL_FILE))

# 筛选层位井点
data_well_purpose_surface_position = (
    data_well_position[data_well_position["Surface"] == SURFACE_NAME]
    .replace(-999, np.nan)
    .dropna(subset=["Sand Thickness"])
    .reset_index(drop=True)
)

# 筛选离群井
data_well_filtered = filter_outlier_wells(data_well_purpose_surface_position, method="iqr")
print(f"筛选后井点数量: {len(data_well_filtered)}")


In [ ]:
# 处理地震属性
attribute_names, _ = identify_attributes(data_seismic_url)
processed_features, stats, report = preprocess_features(
    data=data_seismic_attr,
    attribute_columns=attribute_names,
    missing_values=[-999],
    missing_threshold=0.6,
    outlier_method="iqr",
    outlier_threshold=2.0,
    outlier_treatment="clip",
    verbose=False,
)

attribute_names_processed = list(processed_features.columns)
data_seismic_attr_processed = data_seismic_attr[["X", "Y"]].copy()
for col in processed_features.columns:
    data_seismic_attr_processed[col] = processed_features[col]


In [ ]:
# 限制工区范围
data_seismic_attr_filtered, area_bounds = filter_seismic_by_wells(
    seismic_data=data_seismic_attr_processed,
    well_data=data_well_filtered,
    expansion_factor=1.5,
    plot=False,
)

# 提取井点地震属性
data_well_attr = extract_seismic_attributes_for_wells(
    well_data=data_well_filtered,
    seismic_data=data_seismic_attr_processed,
    max_distance=50,
    num_points=10,
)

# 筛选质量良好的属性
good_attributes, _, _ = filter_anomalous_attributes(
    seismic_data=data_seismic_attr_filtered,
    well_data=data_well_attr,
    common_attributes=attribute_names_processed,
    ratio_threshold=5.0,
    range_ratio_threshold=10.0,
    std_ratio_threshold=10.0,
    verbose=False,
)
print(f"质量良好的属性数量: {len(good_attributes)}")


In [ ]:
# PCA降维（基于地震格点，固定不变）
pca_results = perform_pca_analysis(
    data=data_seismic_attr_filtered,
    attribute_columns=good_attributes,
    variance_threshold=0.9,
    output_dir=output_dir,
)

# 将井点投影到PCA空间
well_features = data_well_attr[pca_results["features_clean"].columns].values
well_features_scaled = pca_results["scaler"].transform(well_features)
well_pca_features = pca_results["pca"].transform(well_features_scaled)

# 准备建模数据
n_components = min(3, well_pca_features.shape[1])
modeling_data = pd.DataFrame()
for i in range(n_components):
    modeling_data[f"PC{i + 1}"] = well_pca_features[:, i]
modeling_data["Sand Thickness"] = data_well_filtered["Sand Thickness"].values
modeling_data["Well_Index"] = data_well_filtered.index.values

print(f"建模数据形状: {modeling_data.shape}")
print(f"PCA主成分数: {n_components}")
print(modeling_data.head())


### 稳定性测试主循环


In [ ]:
def fit_ridge(X, y, alpha=RIDGE_ALPHA):
    """Ridge回归拟合"""
    model = Ridge(alpha=alpha, random_state=RANDOM_SEED)
    model.fit(X, y)
    return {
        "coef": model.coef_.copy(),
        "intercept": model.intercept_,
        "success": True,
    }


def fit_lasso(X, y, alpha=LASSO_ALPHA):
    """Lasso回归拟合"""
    model = Lasso(alpha=alpha, random_state=RANDOM_SEED, max_iter=5000)
    model.fit(X, y)
    return {
        "coef": model.coef_.copy(),
        "intercept": model.intercept_,
        "n_nonzero": np.sum(model.coef_ != 0),
        "success": True,
    }


def fit_sigmoid(data, pc_columns, target_column, virtual_config, pc1_stats):
    """Sigmoid模型拟合"""
    try:
        model = SigmoidModel(data=data, feature_columns=pc_columns, target_column=target_column)

        pc1_min, pc1_max, pc1_median = pc1_stats
        sand_max = data[target_column].max()

        fit_result = model.fit(
            use_features=["PC1"],
            virtual_points_config=virtual_config,
            bounds=(
                [sand_max * 0.2, -10, pc1_min - (pc1_max - pc1_min)],
                [sand_max * 3.0, 10, pc1_max + (pc1_max - pc1_min)],
            ),
            initial_guess=[sand_max * 0.7, 1.0, pc1_median],
            max_iterations=3000,
        )

        if fit_result["success"]:
            return {
                "L": fit_result["params"]["L"],
                "k": fit_result["params"]["k"],
                "x0": fit_result["params"]["x0"],
                "r2": fit_result["r2_score"],
                "success": True,
            }
        else:
            return {"success": False, "error": fit_result.get("error", "Unknown")}
    except Exception as e:
        return {"success": False, "error": str(e)}


In [ ]:
# 存储结果
results = {
    "iteration": [],
    "k_removed": [],
    "removed_wells": [],
    # Ridge
    "ridge_coef_pc1": [],
    "ridge_coef_pc2": [],
    "ridge_coef_pc3": [],
    "ridge_intercept": [],
    # Lasso
    "lasso_coef_pc1": [],
    "lasso_coef_pc2": [],
    "lasso_coef_pc3": [],
    "lasso_intercept": [],
    "lasso_n_nonzero": [],
    # Sigmoid
    "sigmoid_L": [],
    "sigmoid_k": [],
    "sigmoid_x0": [],
    "sigmoid_r2": [],
    "sigmoid_success": [],
}

# 准备特征和目标
pc_columns = [col for col in modeling_data.columns if col.startswith("PC")]
X_full = modeling_data[pc_columns].values
y_full = modeling_data["Sand Thickness"].values
well_indices = modeling_data["Well_Index"].values

# PC1统计量（用于Sigmoid初始化）
pc1_stats_full = (
    modeling_data["PC1"].min(),
    modeling_data["PC1"].max(),
    modeling_data["PC1"].median(),
)

print(f"开始稳定性测试，共 {N_ITERATIONS} 次迭代...")
print("-" * 60)

for iteration in range(N_ITERATIONS):
    # 随机选择删除的井数
    k = np.random.choice(K_RANGE)

    # 随机选择要删除的井
    remove_indices = np.random.choice(len(modeling_data), size=k, replace=False)
    keep_mask = np.ones(len(modeling_data), dtype=bool)
    keep_mask[remove_indices] = False

    # 准备子集数据
    X_subset = X_full[keep_mask]
    y_subset = y_full[keep_mask]
    subset_data = modeling_data[keep_mask].copy().reset_index(drop=True)

    # 记录删除的井
    removed_wells = well_indices[remove_indices].tolist()

    # Ridge 拟合
    ridge_result = fit_ridge(X_subset, y_subset)

    # Lasso 拟合
    lasso_result = fit_lasso(X_subset, y_subset)

    # Sigmoid 拟合
    pc1_stats_subset = (
        subset_data["PC1"].min(),
        subset_data["PC1"].max(),
        subset_data["PC1"].median(),
    )
    sigmoid_result = fit_sigmoid(subset_data, pc_columns, "Sand Thickness", VIRTUAL_CONFIG, pc1_stats_subset)

    # 存储结果
    results["iteration"].append(iteration)
    results["k_removed"].append(k)
    results["removed_wells"].append(str(removed_wells))

    # Ridge
    results["ridge_coef_pc1"].append(ridge_result["coef"][0])
    results["ridge_coef_pc2"].append(ridge_result["coef"][1] if len(ridge_result["coef"]) > 1 else np.nan)
    results["ridge_coef_pc3"].append(ridge_result["coef"][2] if len(ridge_result["coef"]) > 2 else np.nan)
    results["ridge_intercept"].append(ridge_result["intercept"])

    # Lasso
    results["lasso_coef_pc1"].append(lasso_result["coef"][0])
    results["lasso_coef_pc2"].append(lasso_result["coef"][1] if len(lasso_result["coef"]) > 1 else np.nan)
    results["lasso_coef_pc3"].append(lasso_result["coef"][2] if len(lasso_result["coef"]) > 2 else np.nan)
    results["lasso_intercept"].append(lasso_result["intercept"])
    results["lasso_n_nonzero"].append(lasso_result["n_nonzero"])

    # Sigmoid
    if sigmoid_result["success"]:
        results["sigmoid_L"].append(sigmoid_result["L"])
        results["sigmoid_k"].append(sigmoid_result["k"])
        results["sigmoid_x0"].append(sigmoid_result["x0"])
        results["sigmoid_r2"].append(sigmoid_result["r2"])
        results["sigmoid_success"].append(True)
    else:
        results["sigmoid_L"].append(np.nan)
        results["sigmoid_k"].append(np.nan)
        results["sigmoid_x0"].append(np.nan)
        results["sigmoid_r2"].append(np.nan)
        results["sigmoid_success"].append(False)

    if (iteration + 1) % 10 == 0:
        print(f"完成 {iteration + 1}/{N_ITERATIONS} 次迭代")

print("-" * 60)
print("稳定性测试完成!")

In [ ]:
# 转换为DataFrame并保存
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(output_dir, "stability_test_results.csv"), index=False)
print(f"结果已保存到: {os.path.join(output_dir, 'stability_test_results.csv')}")


### 稳定性指标计算


In [ ]:
def calc_stability_metrics(values, name):
    """计算稳定性指标"""
    values = np.array(values)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return {"name": name, "mean": np.nan, "std": np.nan, "cv": np.nan, "range": np.nan}

    mean_val = np.mean(values)
    std_val = np.std(values)
    cv = std_val / abs(mean_val) if abs(mean_val) > 1e-10 else np.inf  # 变异系数
    range_val = np.max(values) - np.min(values)

    return {
        "name": name,
        "mean": mean_val,
        "std": std_val,
        "cv": cv,
        "range": range_val,
        "min": np.min(values),
        "max": np.max(values),
    }


# 计算各模型的稳定性指标
stability_metrics = []

# Ridge
stability_metrics.append(calc_stability_metrics(results_df["ridge_coef_pc1"], "Ridge_PC1"))
stability_metrics.append(calc_stability_metrics(results_df["ridge_coef_pc2"], "Ridge_PC2"))
stability_metrics.append(calc_stability_metrics(results_df["ridge_intercept"], "Ridge_Intercept"))

# Lasso
stability_metrics.append(calc_stability_metrics(results_df["lasso_coef_pc1"], "Lasso_PC1"))
stability_metrics.append(calc_stability_metrics(results_df["lasso_coef_pc2"], "Lasso_PC2"))
stability_metrics.append(calc_stability_metrics(results_df["lasso_intercept"], "Lasso_Intercept"))

# Sigmoid
stability_metrics.append(calc_stability_metrics(results_df["sigmoid_L"], "Sigmoid_L"))
stability_metrics.append(calc_stability_metrics(results_df["sigmoid_k"], "Sigmoid_k"))
stability_metrics.append(calc_stability_metrics(results_df["sigmoid_x0"], "Sigmoid_x0"))

metrics_df = pd.DataFrame(stability_metrics)
metrics_df.to_csv(os.path.join(output_dir, "stability_metrics.csv"), index=False)

print("\n=== 稳定性指标汇总 ===")
print(metrics_df.to_string(index=False))

# Sigmoid拟合成功率
sigmoid_success_rate = results_df["sigmoid_success"].sum() / len(results_df) * 100
print(f"\nSigmoid拟合成功率: {sigmoid_success_rate:.1f}%")

# Sigmoid k符号翻转率
sigmoid_k_values = results_df["sigmoid_k"].dropna()
if len(sigmoid_k_values) > 0:
    k_sign_flips = (sigmoid_k_values * sigmoid_k_values.iloc[0] < 0).sum()
    k_flip_rate = k_sign_flips / len(sigmoid_k_values) * 100
    print(f"Sigmoid k符号翻转率: {k_flip_rate:.1f}%")


### 可视化：参数箱线图


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Ridge 系数
ax = axes[0, 0]
ridge_data = [
    results_df["ridge_coef_pc1"].dropna(),
    results_df["ridge_coef_pc2"].dropna(),
]
ax.boxplot(ridge_data, labels=["PC1系数", "PC2系数"])
ax.set_title("Ridge 回归系数分布", fontsize=14)
ax.set_ylabel("系数值")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

# Lasso 系数
ax = axes[0, 1]
lasso_data = [
    results_df["lasso_coef_pc1"].dropna(),
    results_df["lasso_coef_pc2"].dropna(),
]
ax.boxplot(lasso_data, labels=["PC1系数", "PC2系数"])
ax.set_title("Lasso 回归系数分布", fontsize=14)
ax.set_ylabel("系数值")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

# Sigmoid L
ax = axes[0, 2]
ax.boxplot([results_df["sigmoid_L"].dropna()], labels=["L (最大砂厚)"])
ax.set_title("Sigmoid 参数 L 分布", fontsize=14)
ax.set_ylabel("L 值 (m)")

# Sigmoid k
ax = axes[1, 0]
ax.boxplot([results_df["sigmoid_k"].dropna()], labels=["k (增长率)"])
ax.set_title("Sigmoid 参数 k 分布", fontsize=14)
ax.set_ylabel("k 值")
ax.axhline(y=0, color="red", linestyle="--", alpha=0.7, label="k=0 (方向翻转线)")
ax.legend()

# Sigmoid x0
ax = axes[1, 1]
ax.boxplot([results_df["sigmoid_x0"].dropna()], labels=["x0 (中点位置)"])
ax.set_title("Sigmoid 参数 x0 分布", fontsize=14)
ax.set_ylabel("x0 值 (PC1)")

# 截距对比
ax = axes[1, 2]
intercept_data = [
    results_df["ridge_intercept"].dropna(),
    results_df["lasso_intercept"].dropna(),
]
ax.boxplot(intercept_data, labels=["Ridge", "Lasso"])
ax.set_title("回归截距分布对比", fontsize=14)
ax.set_ylabel("截距值")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "stability_boxplot.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"箱线图已保存到: {os.path.join(output_dir, 'stability_boxplot.png')}")


### 可视化：稳定性雷达图


In [ ]:
def normalize_cv(cv_values, lower_is_better=True):
    """将变异系数归一化到0-1，越稳定分数越高"""
    cv_values = np.array(cv_values)
    cv_values = np.clip(cv_values, 0, 2)  # 限制CV最大为2
    if lower_is_better:
        return 1 - cv_values / 2  # CV=0时分数为1，CV=2时分数为0
    return cv_values / 2


# 准备雷达图数据
models = ["Ridge", "Lasso", "Sigmoid"]
metrics_names = ["PC1系数稳定性", "PC2系数稳定性", "截距/L稳定性", "拟合成功率"]

# 计算各模型的稳定性分数
ridge_scores = [
    normalize_cv([metrics_df[metrics_df["name"] == "Ridge_PC1"]["cv"].values[0]])[0],
    normalize_cv([metrics_df[metrics_df["name"] == "Ridge_PC2"]["cv"].values[0]])[0],
    normalize_cv([metrics_df[metrics_df["name"] == "Ridge_Intercept"]["cv"].values[0]])[0],
    1.0,  # Ridge总是成功
]

lasso_scores = [
    normalize_cv([metrics_df[metrics_df["name"] == "Lasso_PC1"]["cv"].values[0]])[0],
    normalize_cv([metrics_df[metrics_df["name"] == "Lasso_PC2"]["cv"].values[0]])[0],
    normalize_cv([metrics_df[metrics_df["name"] == "Lasso_Intercept"]["cv"].values[0]])[0],
    1.0,  # Lasso总是成功
]

sigmoid_cv_L = metrics_df[metrics_df["name"] == "Sigmoid_L"]["cv"].values[0]
sigmoid_cv_k = metrics_df[metrics_df["name"] == "Sigmoid_k"]["cv"].values[0]
sigmoid_cv_x0 = metrics_df[metrics_df["name"] == "Sigmoid_x0"]["cv"].values[0]

sigmoid_scores = [
    normalize_cv([sigmoid_cv_k])[0],  # k作为主要参数
    normalize_cv([sigmoid_cv_x0])[0],
    normalize_cv([sigmoid_cv_L])[0],
    sigmoid_success_rate / 100,
]

# 绘制雷达图
angles = np.linspace(0, 2 * np.pi, len(metrics_names), endpoint=False).tolist()
angles += angles[:1]  # 闭合

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for scores, model, color in zip(
    [ridge_scores, lasso_scores, sigmoid_scores],
    models,
    ["blue", "green", "red"],
):
    scores_closed = scores + scores[:1]
    ax.plot(angles, scores_closed, "o-", linewidth=2, label=model, color=color)
    ax.fill(angles, scores_closed, alpha=0.25, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title("模型稳定性对比\n(分数越高越稳定)", fontsize=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "stability_radar.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"雷达图已保存到: {os.path.join(output_dir, 'stability_radar.png')}")


### 可视化：参数变化趋势图


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Ridge PC1系数随迭代变化
ax = axes[0]
ax.plot(results_df["iteration"], results_df["ridge_coef_pc1"], "b-", alpha=0.7, label="Ridge")
ax.plot(results_df["iteration"], results_df["lasso_coef_pc1"], "g-", alpha=0.7, label="Lasso")
ax.axhline(y=results_df["ridge_coef_pc1"].mean(), color="blue", linestyle="--", alpha=0.5)
ax.axhline(y=results_df["lasso_coef_pc1"].mean(), color="green", linestyle="--", alpha=0.5)
ax.set_xlabel("实验次数")
ax.set_ylabel("线性系数")
ax.set_title("线性模型：斜率随删井实验的变化", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# Sigmoid k参数随迭代变化
ax = axes[1]
ax.plot(results_df["iteration"], results_df["sigmoid_k"], "r-o", alpha=0.7, markersize=3)
ax.axhline(y=0, color="black", linestyle="-", alpha=0.3)
ax.axhline(y=results_df["sigmoid_k"].mean(), color="red", linestyle="--", alpha=0.5)
ax.fill_between(
    results_df["iteration"],
    results_df["sigmoid_k"].mean() - results_df["sigmoid_k"].std(),
    results_df["sigmoid_k"].mean() + results_df["sigmoid_k"].std(),
    alpha=0.2,
    color="red",
)
ax.set_xlabel("实验次数")
ax.set_ylabel("k (增长率)")
ax.set_title("Sigmoid模型：k参数随删井实验的变化 (阴影区域为±1标准差)", fontsize=14)
ax.grid(True, alpha=0.3)

# Sigmoid L参数随迭代变化
ax = axes[2]
ax.plot(results_df["iteration"], results_df["sigmoid_L"], "m-o", alpha=0.7, markersize=3)
ax.axhline(y=results_df["sigmoid_L"].mean(), color="purple", linestyle="--", alpha=0.5)
ax.fill_between(
    results_df["iteration"],
    results_df["sigmoid_L"].mean() - results_df["sigmoid_L"].std(),
    results_df["sigmoid_L"].mean() + results_df["sigmoid_L"].std(),
    alpha=0.2,
    color="purple",
)
ax.set_xlabel("实验次数")
ax.set_ylabel("L (最大砂厚, m)")
ax.set_title("Sigmoid模型：L参数随删井实验的变化 (阴影区域为±1标准差)", fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "stability_trend.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"趋势图已保存到: {os.path.join(output_dir, 'stability_trend.png')}")


### 可视化：变异系数对比条形图


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# 准备数据
param_names = ["PC1系数/k", "PC2系数/x0", "截距/L"]
ridge_cvs = [
    metrics_df[metrics_df["name"] == "Ridge_PC1"]["cv"].values[0],
    metrics_df[metrics_df["name"] == "Ridge_PC2"]["cv"].values[0],
    metrics_df[metrics_df["name"] == "Ridge_Intercept"]["cv"].values[0],
]
lasso_cvs = [
    metrics_df[metrics_df["name"] == "Lasso_PC1"]["cv"].values[0],
    metrics_df[metrics_df["name"] == "Lasso_PC2"]["cv"].values[0],
    metrics_df[metrics_df["name"] == "Lasso_Intercept"]["cv"].values[0],
]
sigmoid_cvs = [
    metrics_df[metrics_df["name"] == "Sigmoid_k"]["cv"].values[0],
    metrics_df[metrics_df["name"] == "Sigmoid_x0"]["cv"].values[0],
    metrics_df[metrics_df["name"] == "Sigmoid_L"]["cv"].values[0],
]

x = np.arange(len(param_names))
width = 0.25

bars1 = ax.bar(x - width, ridge_cvs, width, label="Ridge", color="steelblue")
bars2 = ax.bar(x, lasso_cvs, width, label="Lasso", color="forestgreen")
bars3 = ax.bar(x + width, sigmoid_cvs, width, label="Sigmoid", color="indianred")

ax.set_ylabel("变异系数 (CV)", fontsize=12)
ax.set_title("模型参数稳定性对比\n(变异系数越小越稳定)", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(param_names, fontsize=11)
ax.legend()
ax.grid(axis="y", alpha=0.3)

# 添加数值标签
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height) and not np.isinf(height):
            ax.annotate(
                f"{height:.2f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=9,
            )

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "stability_cv_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"变异系数对比图已保存到: {os.path.join(output_dir, 'stability_cv_comparison.png')}")


### 可视化：空间预测不确定性地图


In [ ]:
# 准备全工区格点的PCA特征（用于预测）
seismic_features = data_seismic_attr_filtered[pca_results["features_clean"].columns].values
seismic_features_scaled = pca_results["scaler"].transform(seismic_features)
seismic_pca_features = pca_results["pca"].transform(seismic_features_scaled)

# 获取坐标
X_coords = data_seismic_attr_filtered["X"].values
Y_coords = data_seismic_attr_filtered["Y"].values

# 存储每次迭代的预测结果
n_points = len(seismic_pca_features)
ridge_predictions = np.zeros((N_ITERATIONS, n_points))
lasso_predictions = np.zeros((N_ITERATIONS, n_points))
sigmoid_predictions = np.zeros((N_ITERATIONS, n_points))

print("重新计算每次迭代的空间预测...")

# 重新设置随机种子以复现相同的删井序列
np.random.seed(RANDOM_SEED)

for iteration in range(N_ITERATIONS):
    # 复现相同的删井操作
    k = np.random.choice(K_RANGE)
    remove_indices = np.random.choice(len(modeling_data), size=k, replace=False)
    keep_mask = np.ones(len(modeling_data), dtype=bool)
    keep_mask[remove_indices] = False

    X_subset = X_full[keep_mask]
    y_subset = y_full[keep_mask]
    subset_data = modeling_data[keep_mask].copy().reset_index(drop=True)

    # Ridge 预测
    ridge_model = Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED)
    ridge_model.fit(X_subset, y_subset)
    ridge_predictions[iteration] = ridge_model.predict(seismic_pca_features)

    # Lasso 预测
    lasso_model = Lasso(alpha=LASSO_ALPHA, random_state=RANDOM_SEED, max_iter=5000)
    lasso_model.fit(X_subset, y_subset)
    lasso_predictions[iteration] = lasso_model.predict(seismic_pca_features)

    # Sigmoid 预测
    pc1_values = seismic_pca_features[:, 0]
    if results_df.loc[iteration, "sigmoid_success"]:
        L = results_df.loc[iteration, "sigmoid_L"]
        k_sig = results_df.loc[iteration, "sigmoid_k"]
        x0 = results_df.loc[iteration, "sigmoid_x0"]
        sigmoid_predictions[iteration] = L / (1 + np.exp(-k_sig * (pc1_values - x0)))
    else:
        sigmoid_predictions[iteration] = np.nan

    if (iteration + 1) % 10 == 0:
        print(f"  完成 {iteration + 1}/{N_ITERATIONS}")

print("预测完成!")

In [ ]:
# 计算每个格点的预测标准差
ridge_std = np.std(ridge_predictions, axis=0)
lasso_std = np.std(lasso_predictions, axis=0)
sigmoid_std = np.nanstd(sigmoid_predictions, axis=0)

# 计算预测均值（用于参考）
ridge_mean = np.mean(ridge_predictions, axis=0)
lasso_mean = np.mean(lasso_predictions, axis=0)
sigmoid_mean = np.nanmean(sigmoid_predictions, axis=0)

# 计算变异系数（相对不确定性）
ridge_cv_spatial = ridge_std / np.abs(ridge_mean + 1e-10)
lasso_cv_spatial = lasso_std / np.abs(lasso_mean + 1e-10)
sigmoid_cv_spatial = sigmoid_std / np.abs(sigmoid_mean + 1e-10)

print(f"Ridge 预测标准差: min={ridge_std.min():.2f}, max={ridge_std.max():.2f}, mean={ridge_std.mean():.2f}")
print(f"Lasso 预测标准差: min={lasso_std.min():.2f}, max={lasso_std.max():.2f}, mean={lasso_std.mean():.2f}")
print(
    f"Sigmoid 预测标准差: min={np.nanmin(sigmoid_std):.2f}, max={np.nanmax(sigmoid_std):.2f}, mean={np.nanmean(sigmoid_std):.2f}"
)


In [ ]:
# 绘制预测标准差地图
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 统一颜色范围
vmin_std = 0
vmax_std = max(ridge_std.max(), lasso_std.max(), np.nanmax(sigmoid_std))

# 井点位置
well_x = data_well_filtered["X"].values
well_y = data_well_filtered["Y"].values

# Ridge 标准差地图
ax = axes[0, 0]
sc = ax.scatter(X_coords, Y_coords, c=ridge_std, cmap="YlOrRd", s=10, vmin=vmin_std, vmax=vmax_std)
ax.scatter(well_x, well_y, c="blue", s=50, marker="^", edgecolors="white", linewidths=0.5, label="井位")
ax.set_title("Ridge 预测标准差\n(颜色越深 = 越不稳定)", fontsize=14)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="标准差 (m)")

# Lasso 标准差地图
ax = axes[0, 1]
sc = ax.scatter(X_coords, Y_coords, c=lasso_std, cmap="YlOrRd", s=10, vmin=vmin_std, vmax=vmax_std)
ax.scatter(well_x, well_y, c="blue", s=50, marker="^", edgecolors="white", linewidths=0.5, label="井位")
ax.set_title("Lasso 预测标准差\n(颜色越深 = 越不稳定)", fontsize=14)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="标准差 (m)")

# Sigmoid 标准差地图
ax = axes[0, 2]
sc = ax.scatter(X_coords, Y_coords, c=sigmoid_std, cmap="YlOrRd", s=10, vmin=vmin_std, vmax=vmax_std)
ax.scatter(well_x, well_y, c="blue", s=50, marker="^", edgecolors="white", linewidths=0.5, label="井位")
ax.set_title("Sigmoid 预测标准差\n(颜色越深 = 越不稳定)", fontsize=14)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="标准差 (m)")

# 预测均值地图（参考）
vmin_mean = min(ridge_mean.min(), lasso_mean.min(), np.nanmin(sigmoid_mean))
vmax_mean = max(ridge_mean.max(), lasso_mean.max(), np.nanmax(sigmoid_mean))

ax = axes[1, 0]
sc = ax.scatter(X_coords, Y_coords, c=ridge_mean, cmap="viridis", s=10, vmin=vmin_mean, vmax=vmax_mean)
ax.scatter(well_x, well_y, c="red", s=50, marker="^", edgecolors="white", linewidths=0.5, label="井位")
ax.set_title("Ridge 预测均值", fontsize=14)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="砂厚 (m)")

ax = axes[1, 1]
sc = ax.scatter(X_coords, Y_coords, c=lasso_mean, cmap="viridis", s=10, vmin=vmin_mean, vmax=vmax_mean)
ax.scatter(well_x, well_y, c="red", s=50, marker="^", edgecolors="white", linewidths=0.5, label="井位")
ax.set_title("Lasso 预测均值", fontsize=14)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="砂厚 (m)")

ax = axes[1, 2]
sc = ax.scatter(X_coords, Y_coords, c=sigmoid_mean, cmap="viridis", s=10, vmin=vmin_mean, vmax=vmax_mean)
ax.scatter(well_x, well_y, c="red", s=50, marker="^", edgecolors="white", linewidths=0.5, label="井位")
ax.set_title("Sigmoid 预测均值", fontsize=14)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="砂厚 (m)")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "stability_spatial_map.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"空间不确定性地图已保存到: {os.path.join(output_dir, 'stability_spatial_map.png')}")


In [ ]:
# 绘制模型间标准差对比直方图
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(ridge_std, bins=50, alpha=0.5, label=f"Ridge (均值={ridge_std.mean():.2f}m)", color="steelblue")
ax.hist(lasso_std, bins=50, alpha=0.5, label=f"Lasso (均值={lasso_std.mean():.2f}m)", color="forestgreen")
ax.hist(
    sigmoid_std[~np.isnan(sigmoid_std)],
    bins=50,
    alpha=0.5,
    label=f"Sigmoid (均值={np.nanmean(sigmoid_std):.2f}m)",
    color="indianred",
)

ax.set_xlabel("预测标准差 (m)", fontsize=12)
ax.set_ylabel("格点数量", fontsize=12)
ax.set_title("三种模型空间预测不确定性分布\n(标准差越小越稳定)", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "stability_std_histogram.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"标准差直方图已保存到: {os.path.join(output_dir, 'stability_std_histogram.png')}")


In [ ]:
# 汇总空间稳定性指标
print("\n=== 空间预测稳定性汇总 ===")
print(f"\n预测标准差统计 (单位: m):")
print(f"  Ridge:   均值={ridge_std.mean():.3f}, 中位数={np.median(ridge_std):.3f}, 最大={ridge_std.max():.3f}")
print(f"  Lasso:   均值={lasso_std.mean():.3f}, 中位数={np.median(lasso_std):.3f}, 最大={lasso_std.max():.3f}")
print(
    f"  Sigmoid: 均值={np.nanmean(sigmoid_std):.3f}, 中位数={np.nanmedian(sigmoid_std):.3f}, 最大={np.nanmax(sigmoid_std):.3f}"
)

# 计算"高不确定区"占比（标准差 > 均值+1标准差）
threshold_ridge = ridge_std.mean() + ridge_std.std()
threshold_lasso = lasso_std.mean() + lasso_std.std()
threshold_sigmoid = np.nanmean(sigmoid_std) + np.nanstd(sigmoid_std)

high_uncertainty_ridge = (ridge_std > threshold_ridge).sum() / len(ridge_std) * 100
high_uncertainty_lasso = (lasso_std > threshold_lasso).sum() / len(lasso_std) * 100
high_uncertainty_sigmoid = (sigmoid_std > threshold_sigmoid).sum() / np.sum(~np.isnan(sigmoid_std)) * 100

print(f"\n高不确定性区域占比 (标准差 > 均值+1σ):")
print(f"  Ridge:   {high_uncertainty_ridge:.1f}%")
print(f"  Lasso:   {high_uncertainty_lasso:.1f}%")
print(f"  Sigmoid: {high_uncertainty_sigmoid:.1f}%")

### 总结报告


In [ ]:
print("\n" + "=" * 60)
print("稳定性测试总结报告")
print("=" * 60)

print(f"\n实验设置:")
print(f"  - 总井数: {len(modeling_data)}")
print(f"  - 删井数量: {K_RANGE}")
print(f"  - 重复次数: {N_ITERATIONS}")

print(f"\n模型拟合成功率:")
print(f"  - Ridge: 100%")
print(f"  - Lasso: 100%")
print(f"  - Sigmoid: {sigmoid_success_rate:.1f}%")

print(f"\n参数稳定性 (变异系数CV，越小越稳定):")
print(f"\n  Ridge:")
print(f"    - PC1系数 CV: {metrics_df[metrics_df['name'] == 'Ridge_PC1']['cv'].values[0]:.4f}")
print(f"    - PC2系数 CV: {metrics_df[metrics_df['name'] == 'Ridge_PC2']['cv'].values[0]:.4f}")
print(f"    - 截距 CV: {metrics_df[metrics_df['name'] == 'Ridge_Intercept']['cv'].values[0]:.4f}")

print(f"\n  Lasso:")
print(f"    - PC1系数 CV: {metrics_df[metrics_df['name'] == 'Lasso_PC1']['cv'].values[0]:.4f}")
print(f"    - PC2系数 CV: {metrics_df[metrics_df['name'] == 'Lasso_PC2']['cv'].values[0]:.4f}")
print(f"    - 截距 CV: {metrics_df[metrics_df['name'] == 'Lasso_Intercept']['cv'].values[0]:.4f}")

print(f"\n  Sigmoid:")
print(f"    - L (最大砂厚) CV: {metrics_df[metrics_df['name'] == 'Sigmoid_L']['cv'].values[0]:.4f}")
print(f"    - k (增长率) CV: {metrics_df[metrics_df['name'] == 'Sigmoid_k']['cv'].values[0]:.4f}")
print(f"    - x0 (中点) CV: {metrics_df[metrics_df['name'] == 'Sigmoid_x0']['cv'].values[0]:.4f}")

if len(sigmoid_k_values) > 0:
    print(f"    - k符号翻转率: {k_flip_rate:.1f}%")

print("\n" + "=" * 60)
print("输出文件:")
print(f"  - {os.path.join(output_dir, 'stability_test_results.csv')}")
print(f"  - {os.path.join(output_dir, 'stability_metrics.csv')}")
print(f"  - {os.path.join(output_dir, 'stability_boxplot.png')}")
print(f"  - {os.path.join(output_dir, 'stability_radar.png')}")
print(f"  - {os.path.join(output_dir, 'stability_trend.png')}")
print(f"  - {os.path.join(output_dir, 'stability_cv_comparison.png')}")
print("=" * 60)

In [ ]:
# 先用全量数据训练三种模型，得到"基准预测"
print("训练全量数据模型...")

# Ridge 全量
ridge_full = Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED)
ridge_full.fit(X_full, y_full)
ridge_pred_full = ridge_full.predict(seismic_pca_features)

# Lasso 全量
lasso_full = Lasso(alpha=LASSO_ALPHA, random_state=RANDOM_SEED, max_iter=5000)
lasso_full.fit(X_full, y_full)
lasso_pred_full = lasso_full.predict(seismic_pca_features)

# Sigmoid 全量
pc1_values = seismic_pca_features[:, 0]
sigmoid_full_result = fit_sigmoid(modeling_data, pc_columns, "Sand Thickness", VIRTUAL_CONFIG, pc1_stats_full)
if sigmoid_full_result["success"]:
    L_full = sigmoid_full_result["L"]
    k_full = sigmoid_full_result["k"]
    x0_full = sigmoid_full_result["x0"]
    sigmoid_pred_full = L_full / (1 + np.exp(-k_full * (pc1_values - x0_full)))
    print(f"Sigmoid全量模型: L={L_full:.2f}, k={k_full:.4f}, x0={x0_full:.2f}")
else:
    sigmoid_pred_full = np.full_like(pc1_values, np.nan)
    print("Sigmoid全量模型拟合失败")

print("全量模型训练完成!")

In [ ]:
from scipy.stats import pearsonr

best_results = {
    "ridge": {"iteration": -1, "corr": -1, "predictions": None},
    "lasso": {"iteration": -1, "corr": -1, "predictions": None},
    "sigmoid": {"iteration": -1, "corr": -1, "predictions": None},
}

print("\n计算各次删井实验与全量模型的相关性...")

for iteration in range(N_ITERATIONS):
    # Ridge
    ridge_corr, _ = pearsonr(ridge_predictions[iteration], ridge_pred_full)
    if ridge_corr > best_results["ridge"]["corr"]:
        best_results["ridge"]["iteration"] = iteration
        best_results["ridge"]["corr"] = ridge_corr
        best_results["ridge"]["predictions"] = ridge_predictions[iteration].copy()

    # Lasso
    lasso_corr, _ = pearsonr(lasso_predictions[iteration], lasso_pred_full)
    if lasso_corr > best_results["lasso"]["corr"]:
        best_results["lasso"]["iteration"] = iteration
        best_results["lasso"]["corr"] = lasso_corr
        best_results["lasso"]["predictions"] = lasso_predictions[iteration].copy()

    # Sigmoid (跳过失败的)
    if not np.isnan(sigmoid_predictions[iteration]).all() and not np.isnan(sigmoid_pred_full).all():
        valid_mask = ~np.isnan(sigmoid_predictions[iteration]) & ~np.isnan(sigmoid_pred_full)
        if valid_mask.sum() > 10:
            sigmoid_corr, _ = pearsonr(sigmoid_predictions[iteration][valid_mask], sigmoid_pred_full[valid_mask])
            if sigmoid_corr > best_results["sigmoid"]["corr"]:
                best_results["sigmoid"]["iteration"] = iteration
                best_results["sigmoid"]["corr"] = sigmoid_corr
                best_results["sigmoid"]["predictions"] = sigmoid_predictions[iteration].copy()

print("\n=== 最优删井结果 ===")
for model in ["ridge", "lasso", "sigmoid"]:
    it = best_results[model]["iteration"]
    corr = best_results[model]["corr"]
    if it >= 0:
        k_removed = results_df.loc[it, "k_removed"]
        removed_wells = results_df.loc[it, "removed_wells"]
        print(f"{model.upper()}: 第{it}次迭代, 相关系数={corr:.6f}, 删除{k_removed}口井: {removed_wells}")
    else:
        print(f"{model.upper()}: 无有效结果")


In [ ]:
# 输出最优结果到文件
output_best = pd.DataFrame(
    {
        "X": X_coords,
        "Y": Y_coords,
        # 全量模型预测
        "Ridge_Full": ridge_pred_full,
        "Lasso_Full": lasso_pred_full,
        "Sigmoid_Full": sigmoid_pred_full,
        # 最优删井模型预测
        "Ridge_Best": best_results["ridge"]["predictions"],
        "Lasso_Best": best_results["lasso"]["predictions"],
        "Sigmoid_Best": best_results["sigmoid"]["predictions"]
        if best_results["sigmoid"]["predictions"] is not None
        else np.nan,
    }
)

output_file = os.path.join(output_dir, "best_correlation_predictions.csv")
output_best.to_csv(output_file, index=False)
print(f"\n最优结果已保存到: {output_file}")

# 同时保存最优迭代的详细信息
best_info = []
for model in ["ridge", "lasso", "sigmoid"]:
    it = best_results[model]["iteration"]
    if it >= 0:
        info = {
            "model": model,
            "best_iteration": it,
            "correlation": best_results[model]["corr"],
            "k_removed": results_df.loc[it, "k_removed"],
            "removed_wells": results_df.loc[it, "removed_wells"],
        }
        # 添加模型参数
        if model == "ridge":
            info["coef_pc1"] = results_df.loc[it, "ridge_coef_pc1"]
            info["coef_pc2"] = results_df.loc[it, "ridge_coef_pc2"]
            info["intercept"] = results_df.loc[it, "ridge_intercept"]
        elif model == "lasso":
            info["coef_pc1"] = results_df.loc[it, "lasso_coef_pc1"]
            info["coef_pc2"] = results_df.loc[it, "lasso_coef_pc2"]
            info["intercept"] = results_df.loc[it, "lasso_intercept"]
        else:
            info["L"] = results_df.loc[it, "sigmoid_L"]
            info["k"] = results_df.loc[it, "sigmoid_k"]
            info["x0"] = results_df.loc[it, "sigmoid_x0"]
        best_info.append(info)

best_info_df = pd.DataFrame(best_info)
best_info_file = os.path.join(output_dir, "best_correlation_info.csv")
best_info_df.to_csv(best_info_file, index=False)
print(f"最优迭代信息已保存到: {best_info_file}")


In [ ]:
# 可视化：全量 vs 最优删井预测对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, model in enumerate(["ridge", "lasso", "sigmoid"]):
    ax = axes[idx]

    if model == "ridge":
        full_pred = ridge_pred_full
        best_pred = best_results["ridge"]["predictions"]
    elif model == "lasso":
        full_pred = lasso_pred_full
        best_pred = best_results["lasso"]["predictions"]
    else:
        full_pred = sigmoid_pred_full
        best_pred = best_results["sigmoid"]["predictions"]

    if best_pred is not None and not np.isnan(best_pred).all():
        valid = ~np.isnan(full_pred) & ~np.isnan(best_pred)
        ax.scatter(full_pred[valid], best_pred[valid], alpha=0.3, s=1)

        # 1:1线
        lims = [
            min(full_pred[valid].min(), best_pred[valid].min()),
            max(full_pred[valid].max(), best_pred[valid].max()),
        ]
        ax.plot(lims, lims, "r--", label="1:1线")

        corr = best_results[model]["corr"]
        ax.set_title(f"{model.upper()}\n相关系数: {corr:.4f}", fontsize=12)
    else:
        ax.set_title(f"{model.upper()}\n无有效数据", fontsize=12)

    ax.set_xlabel("全量模型预测 (m)")
    ax.set_ylabel("最优删井模型预测 (m)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "best_correlation_scatter.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"\n散点图已保存到: {os.path.join(output_dir, 'best_correlation_scatter.png')}")